## Inicialización

In [1]:
import sys
sys.path.append("..")

In [2]:
from pipeline.ingesta import cargar_indices_desde_bd
from utils.conexionDB import set_db_mode
from utils.graficas import *
from utils.aplicar_whittaker import aplicar_whittaker_series
from pipeline.modulo_fenologico import segmentar_ciclos
import numpy as np
set_db_mode("pruebas")  # <===================IMPORTANTE CAMBIA A LA BD DE PRUEBAS======================================

2026-07-12 13:27:22.652 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager
2026-07-12 13:27:22.653 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager
2026-07-12 13:27:22.654 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager
2026-07-12 13:27:22.656 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager
2026-07-12 13:27:22.656 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager
2026-07-12 13:27:22.657 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager
2026-07-12 13:27:22.658 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager
2026-07-12 13:27:22.659 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager
2026-07-

  📁 Modo BD cambiado a: PRUEBAS (c:\Users\mayno\Desktop\SeminarioInvestigacion\segmentacion_clasificacion_estimacion_maiz_comayagua\notebooks\..\data\pipeline_pruebas.gpkg)


In [3]:
temp_ext = ["2025-01-01","2026-07-11"]

In [4]:
graficar_whittaker_sos(temp_ext[0],temp_ext[1], distancia_min_dias=70, prominencia_min=0.05)

✅  Índices cargados desde BD: 533 fechas × 19 parcelas (2025-01-01 → 2026-07-09).
📈 Suavizando serie temporal para: EVI...
📈 Suavizando serie temporal para: LSWI...

🌾 Calculando FPAR y Factor de Estrés Hídrico Diario (W_scalar)...

✅ Métricas base del VPM calculadas y consolidadas a nivel DIARIO:
   ✔️ Total de Parcelas Procesadas: 19
   ✔️ FPAR Diario (Máx Global): 0.987
   ✔️ W_scalar Diario (Mín/Máx Global): 0.663 / 1.000


interactive(children=(Dropdown(description='🌱 Parcela:', options=('id_0', 'id_1', 'id_10', 'id_11', 'id_12', '…

In [5]:
dfs_indices = cargar_indices_desde_bd(temp_ext[0],temp_ext[1])

✅  Índices cargados desde BD: 533 fechas × 19 parcelas (2025-01-01 → 2026-07-09).


In [6]:
def procesar_indice(df_indice, lambda_param=4000, rango_valido=(-1.0, 1.0)):
    df = df_indice.mask((df_indice < rango_valido[0]) | (df_indice > rango_valido[1]), np.nan)
    rango_diario = pd.date_range(start=df.index.min(), end=df.index.max(), freq="D")
    df = df.reindex(rango_diario)
    df_suavizado = aplicar_whittaker_series({"INDICE": df}, lambda_param=lambda_param)
    return df_suavizado["INDICE"]

df_evi = procesar_indice(dfs_indices["EVI"].copy())
df_lswi = procesar_indice(dfs_indices["LSWI"].copy())

📈 Suavizando serie temporal para: INDICE...
📈 Suavizando serie temporal para: INDICE...


In [7]:
segmentos_por_parcela: dict[int, list[tuple[pd.Timestamp, pd.Timestamp]]] = {}

for col in df_evi.columns:
    try:
        id_parcela = int(col.split("_")[1])
    except (IndexError, ValueError):
        continue

    serie = df_evi[col].dropna()
    if serie.empty:
        continue

    segmentos = segmentar_ciclos(
        serie=serie,
        distancia_min_dias=70,
        prominencia_min=0.05,
    )
    segmentos_por_parcela[id_parcela] = segmentos

In [8]:
sos_por_segmento: dict[int, list[dict]] = {}
ciclos_creados = 0

for id_parcela, segmentos in segmentos_por_parcela.items():
    col = f"id_{id_parcela}"
    if col not in df_evi.columns:
        continue

    lista_resultados = []
    for inicio, fin in segmentos:
        serie_seg = df_evi.loc[inicio:fin, col].dropna()
        if serie_seg.empty:
            continue

        resultado = detectar_sos(
            serie=serie_seg.values,
            fechas=serie_seg.index,
            factor=0.2,
            ventana_busqueda=(inicio, fin),
        )
        resultado["inicio_segmento"] = inicio
        resultado["fin_segmento"] = fin
        lista_resultados.append(resultado)

        sos_fecha = resultado.get("sos_fecha")
        if sos_fecha is not None:
                ciclos_creados += 1

    if lista_resultados:
            sos_por_segmento[id_parcela] = lista_resultados

In [9]:
# Celda — Conteo de ciclos detectados (ya cerrados por diseño de segmentar_ciclos)

resultados_validados = []
for id_parcela, lista_resultados in sos_por_segmento.items():
    for r in lista_resultados:
        if r.get("sos_fecha") is None:
            continue
        r["id_parcela"] = id_parcela
        r["duracion_dias"] = (r["fin_segmento"] - r["sos_fecha"]).days
        resultados_validados.append(r)

df_ciclos_detectados = pd.DataFrame(resultados_validados)

print(f"Total de ciclos detectados con SOS y cierre confirmado: {len(df_ciclos_detectados)}")
print(f"Parcelas distintas representadas: {df_ciclos_detectados['id_parcela'].nunique()}")
print()
print("Distribución de duración de ciclo (días):")
print(df_ciclos_detectados["duracion_dias"].describe())
print()
print(df_ciclos_detectados[["id_parcela", "sos_fecha", "fin_segmento", "duracion_dias"]].sort_values(["id_parcela", "sos_fecha"]))

Total de ciclos detectados con SOS y cierre confirmado: 44
Parcelas distintas representadas: 19

Distribución de duración de ciclo (días):
count     44.000000
mean     128.363636
std       54.227810
min       62.000000
25%       96.500000
50%      114.000000
75%      144.750000
max      312.000000
Name: duracion_dias, dtype: float64

    id_parcela  sos_fecha fin_segmento  duracion_dias
0            0 2025-06-02   2025-10-09            129
1            0 2025-11-06   2026-03-11            125
2            1 2025-05-27   2025-07-31             65
3            1 2025-09-01   2026-03-05            185
22           2 2025-04-27   2025-09-29            155
23           2 2025-12-08   2026-04-03            116
24           3 2025-04-27   2025-09-26            152
25           3 2025-12-08   2026-04-01            114
26           4 2025-04-27   2025-09-28            154
27           4 2025-12-12   2026-04-08            117
28           5 2025-04-27   2025-09-27            153
29           5 2

In [12]:
# Celda — Tabla de correcciones manuales (override sobre el cruce automático)

correcciones_manuales = pd.DataFrame([
    {
        "id_parcela": 10,
        "sos_confirmado": pd.Timestamp("2025-08-26"),
        "fin_confirmado": pd.Timestamp("2025-12-22"),
        "origen_etiqueta": "confirmado_campo_fecha_invalida",
        "justificacion": (
            "Fecha de siembra/cosecha reportada de memoria por el productor"
            "Se usa el segmento detectado por segmentar_ciclos (campana limpia sin ruido "
            "confirmada visualmente). HIPOTESIS: de los 3 segmentos detectados para esta "
            "parcela, se elige este por ser el que mejor calza con la ventana calendario "
            "de postrera (ago-mar) y con la duracion reportada (~117d vs 115d detectados). "
        ),
    },
    {
        "id_parcela": 11,
        "sos_confirmado": pd.Timestamp("2025-08-21"),
        "fin_confirmado": pd.Timestamp("2025-12-22"),
        "origen_etiqueta": "confirmado_campo_fecha_invalida",
        "justificacion": (
            "Mismo caso que id_10 (mismo productor, misma fecha reportada de memoria). "
            "Segmento elegido por ventana postrera + duracion consistente (112d). "
            "PENDIENTE: confirmar contra grafica propia de id_11."
        ),
    },
    {
        "id_parcela": 14,
        "sos_confirmado": pd.Timestamp("2025-06-07"),
        "fin_confirmado": pd.Timestamp("2025-10-17"),
        "origen_etiqueta": "ciclo_sin_valle_cierre",
        "justificacion": (
            "Ciclo sin valle de cierre detectado, pero con valores claramente llegando a tierra desnuda."
        ),
    },
    {
        "id_parcela": 15,
        "sos_confirmado": pd.Timestamp("2025-07-14"),
        "fin_confirmado": pd.Timestamp("2025-10-31"),  # truncado a cosecha de campo, no al cierre detectado (312d)
        "origen_etiqueta": "confirmado_dicta_truncado_manual",
        "justificacion": (
            "Segmento detectado (312d) fusiona multiples eventos por valle de baja "
            "prominencia cerca de la fecha de cosecha reportada. Se trunca manualmente "
            "usando fecha_cosecha_campo como cierre, en vez del cierre real detectado."
        ),
    },
    {
        "id_parcela": 16,
        "sos_confirmado": pd.Timestamp("2025-07-07"),
        "fin_confirmado": pd.Timestamp("2025-11-22"),  # truncado a cosecha de campo, no al cierre detectado (242d)
        "origen_etiqueta": "confirmado_dicta_truncado_manual",
        "justificacion": "Mismo caso que id_15: se trunca a fecha de cosecha de campo.",
    },
    {
        "id_parcela": 18,
        "sos_confirmado": pd.Timestamp("2025-04-16"),
        "fin_confirmado": pd.Timestamp("2025-08-19"),
        "origen_etiqueta": "confirmado_campo_fecha_aproximada",
        "justificacion": (
            "Fecha reportada de memoria por tercero (no el productor). Se acepta el "
            "segmento detectado (114d) tal cual, desfase de 35d atribuible a imprecision "
            "del dato, no a error de deteccion."
        ),
    },
])

correcciones_manuales["duracion_dias"] = (
    correcciones_manuales["fin_confirmado"] - correcciones_manuales["sos_confirmado"]
).dt.days

excluidos_esta_ronda = pd.DataFrame([
    {"id_parcela": 13, "motivo": "sin senal de EVI detectable para el ciclo de elote reportado"},
])

print("Correcciones manuales aplicadas:")
display(correcciones_manuales[["id_parcela", "sos_confirmado", "fin_confirmado", "duracion_dias", "origen_etiqueta"]])
print("\nExcluidos de esta ronda:")
display(excluidos_esta_ronda)

Correcciones manuales aplicadas:


,id_parcela,sos_confirmado,fin_confirmado,duracion_dias,origen_etiqueta
0,10,2025-08-26,2025-12-22,118,confirmado_campo_fecha_invalida
1,11,2025-08-21,2025-12-22,123,confirmado_campo_fecha_invalida
2,14,2025-06-07,2025-10-17,132,ciclo_sin_valle_cierre
3,15,2025-07-14,2025-10-31,109,confirmado_dicta_truncado_manual
4,16,2025-07-07,2025-11-22,138,confirmado_dicta_truncado_manual
5,18,2025-04-16,2025-08-19,125,confirmado_campo_fecha_aproximada



Excluidos de esta ronda:


,id_parcela,motivo
0,13,sin senal de EVI detectable para el ciclo de e...


In [13]:
# Celda — Consolidar tabla final de ciclos (detectados + corregidos), sin duplicar traslapes

tabla_ciclos_final = df_ciclos_detectados.copy()
tabla_ciclos_final["es_confirmado_maiz"] = False
tabla_ciclos_final["origen_etiqueta"] = "no_confirmado"

filas_finales = []

for _, corr in correcciones_manuales.iterrows():
    id_p = corr["id_parcela"]

    # Elimina cualquier segmento original de esta parcela que se traslape
    # con la ventana corregida (evita contar el mismo período dos veces)
    mask_traslape = (
        (tabla_ciclos_final["id_parcela"] == id_p)
        & (tabla_ciclos_final["sos_fecha"] < corr["fin_confirmado"])
        & (tabla_ciclos_final["fin_segmento"] > corr["sos_confirmado"])
    )
    n_eliminados = mask_traslape.sum()
    if n_eliminados > 0:
        print(f"id_parcela {id_p}: eliminando {n_eliminados} segmento(s) original(es) que traslapan con la corrección")
    tabla_ciclos_final = tabla_ciclos_final[~mask_traslape]

    filas_finales.append({
        "id_parcela": id_p,
        "sos_fecha": corr["sos_confirmado"],
        "fin_segmento": corr["fin_confirmado"],
        "duracion_dias": corr["duracion_dias"],
        "es_confirmado_maiz": True,
        "origen_etiqueta": corr["origen_etiqueta"],
    })

tabla_ciclos_final = pd.concat(
    [tabla_ciclos_final, pd.DataFrame(filas_finales)],
    ignore_index=True,
).sort_values(["id_parcela", "sos_fecha"]).reset_index(drop=True)

# Excluye explícitamente el caso 13 (sin señal detectable) si algún segmento suyo sigue en la tabla
tabla_ciclos_final = tabla_ciclos_final[
    ~((tabla_ciclos_final["id_parcela"] == 13) & (tabla_ciclos_final["origen_etiqueta"] == "no_confirmado") & False)
]  # placeholder: sus segmentos originales SÍ se conservan como no-confirmados (corpus general), solo no se marcan como maíz

print(f"\nTotal de ciclos en el corpus final: {len(tabla_ciclos_final)}")
print(f"Confirmados de maíz: {tabla_ciclos_final['es_confirmado_maiz'].sum()}")
print(f"Parcelas distintas: {tabla_ciclos_final['id_parcela'].nunique()}")
tabla_ciclos_final[tabla_ciclos_final["es_confirmado_maiz"]][
    ["id_parcela", "sos_fecha", "fin_segmento", "duracion_dias", "origen_etiqueta"]
]

id_parcela 10: eliminando 1 segmento(s) original(es) que traslapan con la corrección
id_parcela 11: eliminando 1 segmento(s) original(es) que traslapan con la corrección
id_parcela 15: eliminando 1 segmento(s) original(es) que traslapan con la corrección
id_parcela 16: eliminando 1 segmento(s) original(es) que traslapan con la corrección
id_parcela 18: eliminando 1 segmento(s) original(es) que traslapan con la corrección

Total de ciclos en el corpus final: 45
Confirmados de maíz: 6
Parcelas distintas: 19


,id_parcela,sos_fecha,fin_segmento,duracion_dias,origen_etiqueta
27,10,2025-08-26,2025-12-22,118,confirmado_campo_fecha_invalida
30,11,2025-08-21,2025-12-22,123,confirmado_campo_fecha_invalida
37,14,2025-06-07,2025-10-17,132,ciclo_sin_valle_cierre
38,15,2025-07-14,2025-10-31,109,confirmado_dicta_truncado_manual
40,16,2025-07-07,2025-11-22,138,confirmado_dicta_truncado_manual
42,18,2025-04-16,2025-08-19,125,confirmado_campo_fecha_aproximada


In [13]:
# # ==============================================================================
# # CELDA 1: ASIGNACIÓN DE SUBTIPOS DE MAÍZ POR CICLO ESPECÍFICO
# # ==============================================================================

# # 1. Define aquí las combinaciones exactas de Parcela + Fecha de tus datos de campo.
# # Reemplaza los IDs y las fechas con tus datos reales.
# ciclos_confirmados_elote = [
#     {"id_parcela": 15, "año": 2025, "mes": 6},
#     {"id_parcela": 16, "año": 2025, "mes": 7}
# ]

# ciclos_confirmados_grano = [
#     {"id_parcela": 10, "año": 2025, "mes": 8},
#     {"id_parcela": 11, "año": 2025, "mes": 8},
#     {"id_parcela": 14, "año": 2025, "mes": 6},
#     {"id_parcela": 18, "año": 2025, "mes": 4}
# ]

# # 2. Inicializar la columna de control en tu tabla consolidada
# tabla_ciclos_final["subtipo_maiz"] = "desconocido"

# # 3. Asignar 'elote' solo a los ciclos que coinciden en ID, año y mes del SOS
# for ref in ciclos_confirmados_elote:
#     mask = (
#         (tabla_ciclos_final["id_parcela"] == ref["id_parcela"]) & 
#         (tabla_ciclos_final["sos_fecha"].dt.year == ref["año"]) & 
#         (tabla_ciclos_final["sos_fecha"].dt.month == ref["mes"]) &
#         (tabla_ciclos_final["es_confirmado_maiz"] == True)
#     )
#     tabla_ciclos_final.loc[mask, "subtipo_maiz"] = "elote"

# # 4. Asignar 'grano' solo a los ciclos que coinciden en ID, año y mes del SOS
# for ref in ciclos_confirmados_grano:
#     mask = (
#         (tabla_ciclos_final["id_parcela"] == ref["id_parcela"]) & 
#         (tabla_ciclos_final["sos_fecha"].dt.year == ref["año"]) & 
#         (tabla_ciclos_final["sos_fecha"].dt.month == ref["mes"]) &
#         (tabla_ciclos_final["es_confirmado_maiz"] == True)
#     )
#     tabla_ciclos_final.loc[mask, "subtipo_maiz"] = "grano"

# # Conteo de verificación
# print("Conteo de ciclos por subtipo de maíz asignado:")
# print(tabla_ciclos_final["subtipo_maiz"].value_counts())

In [14]:
# ==============================================================================
# CELDA 1 (adaptada): PATRÓN MAESTRO DE GRANO — SOLO EVI
# ==============================================================================

import numpy as np
import pandas as pd

# Patrón principal: los 4 confirmados de emergencia rápida (subgrupo A + id_18)
ciclos_confirmados_grano = [
    {"id_parcela": 10, "año": 2025, "mes": 8},
    {"id_parcela": 11, "año": 2025, "mes": 8},
    {"id_parcela": 14, "año": 2025, "mes": 6},
    {"id_parcela": 18, "año": 2025, "mes": 4},
]

# 15 y 16 quedan aparte, como validación held-out (variedad de ciclo lento) —
# NO entran al patrón maestro, NO se tratan como incógnitas
ciclos_validacion_lenta = [
    {"id_parcela": 15, "año": 2025, "mes": 7},
    {"id_parcela": 16, "año": 2025, "mes": 7},
]

tabla_ciclos_final["subtipo_maiz"] = "desconocido"

for ref in ciclos_confirmados_grano:
    mask = (
        (tabla_ciclos_final["id_parcela"] == ref["id_parcela"]) &
        (tabla_ciclos_final["sos_fecha"].dt.year == ref["año"]) &
        (tabla_ciclos_final["sos_fecha"].dt.month == ref["mes"]) &
        (tabla_ciclos_final["es_confirmado_maiz"] == True)
    )
    tabla_ciclos_final.loc[mask, "subtipo_maiz"] = "grano"

for ref in ciclos_validacion_lenta:
    mask = (
        (tabla_ciclos_final["id_parcela"] == ref["id_parcela"]) &
        (tabla_ciclos_final["sos_fecha"].dt.year == ref["año"]) &
        (tabla_ciclos_final["sos_fecha"].dt.month == ref["mes"]) &
        (tabla_ciclos_final["es_confirmado_maiz"] == True)
    )
    tabla_ciclos_final.loc[mask, "subtipo_maiz"] = "grano_lento_validacion"

ciclos_referencia_grano = tabla_ciclos_final[tabla_ciclos_final["subtipo_maiz"] == "grano"]

def extraer_matrices_por_ciclo(df_indice, tabla_filtrada, ventana=60):
    matrices, ids_usados = [], []
    for idx, ciclo in tabla_filtrada.iterrows():
        id_p = ciclo["id_parcela"]
        col = f"id_{id_p}"
        sos = ciclo["sos_fecha"]
        rango_fechas = pd.date_range(start=sos, periods=ventana + 1, freq="D")
        try:
            serie_alineada = df_indice.loc[rango_fechas, col].values
            matrices.append(serie_alineada)
            ids_usados.append(id_p)
        except KeyError:
            print(f"⚠️ Sin datos para '{col}' en el rango del SOS {sos.strftime('%Y-%m-%d')}")
            continue
    return np.array(matrices), ids_usados

# Solo EVI — se elimina por completo la extracción/uso de LSWI en este módulo
matriz_evi_grano, ids_grano = extraer_matrices_por_ciclo(df_evi, ciclos_referencia_grano, ventana=60)

mu_evi_grano = np.mean(matriz_evi_grano, axis=0)
sigma_evi_grano = np.std(matriz_evi_grano, axis=0)

print(f"Ciclos en patrón maestro de grano: {len(ids_grano)} -> {ids_grano}")
print(f"Dimensión matriz EVI: {matriz_evi_grano.shape}")
print(f"⚠️ Nota: con N={len(ids_grano)}, sigma_evi_grano es poco confiable dia a dia — "
      f"úsalo con cautela si en el futuro decides construir bandas de tolerancia sobre EVI.")

Ciclos en patrón maestro de grano: 4 -> [10, 11, 14, 18]
Dimensión matriz EVI: (4, 61)
⚠️ Nota: con N=4, sigma_evi_grano es poco confiable dia a dia — úsalo con cautela si en el futuro decides construir bandas de tolerancia sobre EVI.


In [ ]:
# ==============================================================================
# CELDA 2 (nueva): CALIBRACIÓN LOOCV DEL UMBRAL DE CORRELACIÓN
# ==============================================================================
from scipy.stats import pearsonr

def correlacion_truncada(id_parcela, sos, t_actual, mu_ref, df_evi):
    col = f"id_{id_parcela}"
    rango = pd.date_range(start=sos, periods=t_actual + 1, freq="D")
    try:
        serie_obs = df_evi.loc[rango, col].values
    except KeyError:
        return np.nan
    ref_trunc = mu_ref[: t_actual + 1]
    if np.std(serie_obs) == 0:
        return 0.0
    r, _ = pearsonr(serie_obs, ref_trunc)
    return 0.0 if np.isnan(r) else r

DIAS_A_EVALUAR = [30, 42, 60]

resultados_loocv = []

# # --- LOOCV sobre los 4 de grano: cada uno se compara contra la media de los OTROS 3 ---
# for i, id_p in enumerate(ids_grano):
#     ciclo = ciclos_referencia_grano.iloc[i]
#     matriz_sin_i = np.delete(matriz_evi_grano, i, axis=0)
#     mu_sin_i = np.mean(matriz_sin_i, axis=0)
#     for t in DIAS_A_EVALUAR:
#         r = correlacion_truncada(id_p, ciclo["sos_fecha"], t, mu_sin_i, df_evi)
#         resultados_loocv.append({"id_parcela": id_p, "grupo": "grano_LOOCV", "dia": t, "r": r})

# # --- Validación held-out: 15 y 16 contra el patrón COMPLETO de grano (nunca vieron estos casos) ---
# # --- Validación held-out: 15 y 16 contra el patrón COMPLETO de grano ---     
# for ref in ciclos_validacion_lenta:
#     # 1. Aplicamos la máscara buscando por ID de parcela
#     mask = (tabla_ciclos_final["id_parcela"] == ref["id_parcela"])
    
#     # NOTA DE SEGURIDAD: Quitamos el filtro estricto de "subtipo_maiz" temporalmente 
#     # para asegurar que encuentre la parcela, sin importar el texto de su etiqueta.
#     sub_df = tabla_ciclos_final[mask]
    
#     # 2. Control defensivo: ¿El DataFrame tiene filas?
#     if sub_df.empty:
#         print(f"⚠️ Alerta: No se encontró la parcela ID {ref['id_parcela']} en 'tabla_ciclos_final'. Saltando...")
#         continue
        
#     # 3. Si sobrevivió al control, ahora sí extraemos la primera fila de forma segura
#     fila = sub_df.iloc[0]
#     for t in DIAS_A_EVALUAR:
#         r = correlacion_truncada(ref["id_parcela"], fila["sos_fecha"], t, mu_evi_grano, df_evi)
#         resultados_loocv.append({"id_parcela": ref["id_parcela"], "grupo": "grano_lento_heldout", "dia": t, "r": r})

# df_calibracion = pd.DataFrame(resultados_loocv)
# print(df_calibracion.pivot(index="id_parcela", columns="dia", values="r").round(3))
# print("\nResumen por grupo y día:")
# print(df_calibracion.groupby(["grupo", "dia"])["r"].describe()[["min", "mean", "max"]].round(3))

dia            30     42     60
id_parcela                     
10          0.990  0.987  0.991
11          0.993  0.991  0.993
14          0.991  0.985  0.984
15          0.951  0.929  0.905
16          0.996  0.993  0.988
18          0.995  0.991  0.986

Resumen por grupo y día:
                           min   mean    max
grupo               dia                     
grano_LOOCV         30   0.990  0.992  0.995
                    42   0.985  0.988  0.991
                    60   0.984  0.988  0.993
grano_lento_heldout 30   0.951  0.974  0.996
                    42   0.929  0.961  0.993
                    60   0.905  0.947  0.988


In [16]:
# Celda 2 (corregida) — con diagnóstico explícito en vez de .iloc[0] ciego

# Primero, verifica que la Celda 1 adaptada sí corrió y dejó la columna bien poblada
print(tabla_ciclos_final["subtipo_maiz"].value_counts())
print()

resultados_loocv = []

for i, id_p in enumerate(ids_grano):
    ciclo = ciclos_referencia_grano.iloc[i]
    matriz_sin_i = np.delete(matriz_evi_grano, i, axis=0)
    mu_sin_i = np.mean(matriz_sin_i, axis=0)
    for t in DIAS_A_EVALUAR:
        r = correlacion_truncada(id_p, ciclo["sos_fecha"], t, mu_sin_i, df_evi)
        resultados_loocv.append({"id_parcela": id_p, "grupo": "grano_LOOCV", "dia": t, "r": r})

for ref in ciclos_validacion_lenta:
    match = tabla_ciclos_final[
        (tabla_ciclos_final["id_parcela"] == ref["id_parcela"])
        & (tabla_ciclos_final["subtipo_maiz"] == "grano_lento_validacion")
    ]
    if match.empty:
        print(f"⚠️ No se encontró fila para id_parcela={ref['id_parcela']} "
              f"con año={ref['año']}, mes={ref['mes']} y subtipo_maiz='grano_lento_validacion'. "
              f"Revisa que la Celda 1 adaptada se haya corrido DESPUÉS de la última "
              f"actualización de correcciones_manuales.")
        continue

    fila = match.iloc[0]
    for t in DIAS_A_EVALUAR:
        r = correlacion_truncada(ref["id_parcela"], fila["sos_fecha"], t, mu_evi_grano, df_evi)
        resultados_loocv.append({"id_parcela": ref["id_parcela"], "grupo": "grano_lento_heldout", "dia": t, "r": r})

df_calibracion = pd.DataFrame(resultados_loocv)
print(df_calibracion.pivot(index="id_parcela", columns="dia", values="r").round(3))
print("\nResumen por grupo y día:")
print(df_calibracion.groupby(["grupo", "dia"])["r"].describe()[["min", "mean", "max"]].round(3))

subtipo_maiz
desconocido               39
grano                      4
grano_lento_validacion     2
Name: count, dtype: int64

dia            30     42     60
id_parcela                     
10          0.990  0.987  0.991
11          0.993  0.991  0.993
14          0.991  0.985  0.984
15          0.951  0.929  0.905
16          0.996  0.993  0.988
18          0.995  0.991  0.986

Resumen por grupo y día:
                           min   mean    max
grupo               dia                     
grano_LOOCV         30   0.990  0.992  0.995
                    42   0.985  0.988  0.991
                    60   0.984  0.988  0.993
grano_lento_heldout 30   0.951  0.974  0.996
                    42   0.929  0.961  0.993
                    60   0.905  0.947  0.988


## Clasificador Correlacion Pearson V1

In [ ]:
# ==============================================================================
# CELDA 3 (adaptada): SCORE DE FORMA — SOLO EVI, SIN CLASIFICACIÓN DURA
# ==============================================================================

def evaluar_parcela_forma_evi(id_parcela, fecha_sos, t_actual, df_evi, mu_ref):
    col = f"id_{id_parcela}"
    rango_fechas = pd.date_range(start=fecha_sos, periods=t_actual + 1, freq="D")

    try:
        evi_obs = df_evi.loc[rango_fechas, col].values
    except KeyError:
        return {"score_similitud": np.nan, "estado": "sin_datos"}

    ref_trunc = mu_ref[: t_actual + 1]

    if np.std(evi_obs) == 0:
        r = 0.0
    else:
        r, _ = pearsonr(evi_obs, ref_trunc)
    r = 0.0 if np.isnan(r) else r

    return {"score_similitud": max(0.0, r * 100), "estado": "evaluado"}

In [18]:
# ==============================================================================
# CELDA 4 (adaptada): PANEL DE OPERADOR — SIN ETIQUETA DURA, SOLO RANKING
# ==============================================================================

df_incognitas = tabla_ciclos_final[tabla_ciclos_final["subtipo_maiz"] == "desconocido"]

DIA_ACTUAL_VENTANA = 42

resultados_operador = []
for idx, fila in df_incognitas.iterrows():
    res = evaluar_parcela_forma_evi(
        id_parcela=fila["id_parcela"],
        fecha_sos=fila["sos_fecha"],
        t_actual=DIA_ACTUAL_VENTANA,
        df_evi=df_evi,
        mu_ref=mu_evi_grano,
    )
    resultados_operador.append({
        "id_parcela": fila["id_parcela"],
        "sos_fecha": fila["sos_fecha"].strftime("%Y-%m-%d"),
        "duracion_dias": fila["duracion_dias"],
        "score_similitud": res["score_similitud"],
        "estado": res["estado"],
    })

df_dashboard = pd.DataFrame(resultados_operador).sort_values("score_similitud", ascending=False)
df_dashboard

,id_parcela,sos_fecha,duracion_dias,score_similitud,estado
14,7,2025-03-01,105,99.878064,evaluado
27,10,2026-02-01,109,99.795033,evaluado
18,8,2025-02-21,112,99.599031,evaluado
28,11,2025-03-27,147,99.584363,evaluado
19,8,2025-07-03,95,99.576890,evaluado
25,9,2026-02-04,95,99.430077,evaluado
11,5,2025-12-13,119,99.402342,evaluado
22,9,2025-02-24,102,99.260330,evaluado
29,11,2026-01-20,112,99.213941,evaluado
23,9,2025-06-25,97,99.130516,evaluado


In [19]:
# Celda de control — probar Pearson contra ciclos claramente NO parecidos a maíz

ids_control_negativo = [
    {"id_parcela": 12, "sos_fecha": pd.Timestamp("2025-04-02")},  # ciclo de 300 días, sospechoso
    {"id_parcela": 13, "sos_fecha": pd.Timestamp("2025-12-24")},  # ciclo plano, pendiente_verdeo=0.0004
    {"id_parcela": 0, "sos_fecha": pd.Timestamp("2025-06-02")},   # duración corta, pendiente baja (0.0006)
]

for ctrl in ids_control_negativo:
    for t in DIAS_A_EVALUAR:
        r = correlacion_truncada(ctrl["id_parcela"], ctrl["sos_fecha"], t, mu_evi_grano, df_evi)
        print(f"id_{ctrl['id_parcela']}  dia={t}  r={r:.3f}")

id_12  dia=30  r=0.988
id_12  dia=42  r=0.978
id_12  dia=60  r=0.969
id_13  dia=30  r=0.958
id_13  dia=42  r=0.894
id_13  dia=60  r=0.312
id_0  dia=30  r=0.982
id_0  dia=42  r=0.739
id_0  dia=60  r=-0.223


## Clasificador Dual Pearson Pendiente de Verdeo

In [ ]:
# Celda — Score compuesto: pendiente de verdeo (magnitud) x correlación de forma (Pearson)

def pendiente_verdeo_truncada(id_parcela, sos, t_actual, df_evi, dia_ini=5):
    col = f"id_{id_parcela}"
    dia_fin = min(t_actual - 5, t_actual)  # usa el punto más reciente disponible dentro de la ventana
    if dia_fin <= dia_ini:
        return np.nan
    rango = pd.date_range(start=sos, periods=t_actual + 1, freq="D")
    try:
        serie = df_evi.loc[rango, col].values
    except KeyError:
        return np.nan
    dias = np.arange(len(serie))
    evi_ini = np.interp(dia_ini, dias, serie)
    evi_fin = np.interp(dia_fin, dias, serie)
    return (evi_fin - evi_ini) / (dia_fin - dia_ini)


# Pendiente de referencia (grano), truncada al mismo día, para normalizar la magnitud observada
def evaluar_score_compuesto(id_parcela, sos, t_actual, df_evi, mu_ref, pendiente_ref):
    r = correlacion_truncada(id_parcela, sos, t_actual, mu_ref, df_evi)
    pend_obs = pendiente_verdeo_truncada(id_parcela, sos, t_actual, df_evi)

    if np.isnan(r) or np.isnan(pend_obs):
        return {"r_forma": np.nan, "pendiente_obs": np.nan, "ratio_pendiente": np.nan, "score_compuesto": np.nan}

    # Ratio de magnitud, capado en 1.0 (una pendiente MÁS agresiva que el maíz no debe penalizar)
    ratio_pendiente = min(1.0, max(0.0, pend_obs / pendiente_ref)) if pendiente_ref > 0 else 0.0

    score_compuesto = max(0.0, r) * ratio_pendiente * 100

    return {"r_forma": r, "pendiente_obs": pend_obs, "ratio_pendiente": ratio_pendiente, "score_compuesto": score_compuesto}

# Pendiente de referencia: la del propio patrón maestro, truncada a cada día evaluado
pendientes_ref_por_dia = {}
for t in DIAS_A_EVALUAR:
    dia_ini, dia_fin = 5, min(t - 5, t)
    dias_ref = np.arange(len(mu_evi_grano))
    evi_ini = np.interp(dia_ini, dias_ref, mu_evi_grano)
    evi_fin = np.interp(dia_fin, dias_ref, mu_evi_grano)
    pendientes_ref_por_dia[t] = (evi_fin - evi_ini) / (dia_fin - dia_ini)

print("Pendiente de referencia (patrón grano) por día:", pendientes_ref_por_dia)
print()

casos_prueba = (
    [{"id_parcela": id_p, "sos_fecha": ciclos_referencia_grano[ciclos_referencia_grano["id_parcela"] == id_p]["sos_fecha"].iloc[0], "grupo": "grano_confirmado"} for id_p in ids_grano]
    + [{"id_parcela": 15, "sos_fecha": pd.Timestamp("2025-07-14"), "grupo": "lento_heldout"},
       {"id_parcela": 16, "sos_fecha": pd.Timestamp("2025-07-26"), "grupo": "lento_heldout"}]
    + [{"id_parcela": 12, "sos_fecha": pd.Timestamp("2025-04-02"), "grupo": "control_negativo"},
       {"id_parcela": 13, "sos_fecha": pd.Timestamp("2025-12-24"), "grupo": "control_negativo"},
       {"id_parcela": 0, "sos_fecha": pd.Timestamp("2025-06-02"), "grupo": "control_negativo"}]
)

filas = []
for caso in casos_prueba:
    for t in DIAS_A_EVALUAR:
        res = evaluar_score_compuesto(caso["id_parcela"], caso["sos_fecha"], t, df_evi, mu_evi_grano, pendientes_ref_por_dia[t])
        filas.append({"id_parcela": caso["id_parcela"], "grupo": caso["grupo"], "dia": t, **res})

df_score_compuesto = pd.DataFrame(filas)
df_score_compuesto.pivot_table(index=["grupo", "id_parcela"], columns="dia", values="score_compuesto").round(1)

Pendiente de referencia (patrón grano) por día: {30: np.float64(0.007487738713451206), 42: np.float64(0.007711127669990421), 60: np.float64(0.006219130939448286)}



dia                            30    42    60
grupo            id_parcela                  
control_negativo 0           94.7  33.7   0.0
                 12          82.0  65.2  54.9
                 13          13.0   8.0   0.4
grano_confirmado 10          74.1  88.8  91.9
                 11          89.3  99.5  99.6
                 14          99.8  99.5  99.4
                 18          42.6  36.8  40.6
lento_heldout    15          10.8  23.4  43.5
                 16          57.8  46.9  42.3

## Clasificador Dual v2

In [21]:
# Celda — Recalcular ratio de pendiente usando rango real (min-max) de los confirmados, no un solo promedio

# Primero, calcular la pendiente truncada de CADA uno de los 4 confirmados de grano, por día
pendientes_confirmados_por_dia = {t: [] for t in DIAS_A_EVALUAR}
for id_p in ids_grano:
    sos = ciclos_referencia_grano[ciclos_referencia_grano["id_parcela"] == id_p]["sos_fecha"].iloc[0]
    for t in DIAS_A_EVALUAR:
        p = pendiente_verdeo_truncada(id_p, sos, t, df_evi)
        pendientes_confirmados_por_dia[t].append(p)

rango_pendiente_por_dia = {
    t: (min(vals), max(vals)) for t, vals in pendientes_confirmados_por_dia.items()
}
print("Rango de pendiente (min, max) de los 4 confirmados de grano, por día:")
print(rango_pendiente_por_dia)


def evaluar_score_compuesto_v2(id_parcela, sos, t_actual, df_evi, mu_ref, rango_pendiente):
    r = correlacion_truncada(id_parcela, sos, t_actual, mu_ref, df_evi)
    pend_obs = pendiente_verdeo_truncada(id_parcela, sos, t_actual, df_evi)

    if np.isnan(r) or np.isnan(pend_obs):
        return {"r_forma": np.nan, "pendiente_obs": np.nan, "ratio_pendiente": np.nan, "score_compuesto": np.nan}

    p_min, p_max = rango_pendiente
    # min-max con un margen de tolerancia del 20% por debajo del mínimo observado,
    # para no penalizar de golpe a un confirmado ligeramente más lento que tus 4 muestras
    p_min_tolerante = p_min * 0.8
    if p_max <= p_min_tolerante:
        ratio_pendiente = 1.0
    else:
        ratio_pendiente = min(1.0, max(0.0, (pend_obs - p_min_tolerante) / (p_max - p_min_tolerante)))

    score_compuesto = max(0.0, r) * ratio_pendiente * 100
    return {"r_forma": r, "pendiente_obs": pend_obs, "ratio_pendiente": ratio_pendiente, "score_compuesto": score_compuesto}


filas_v2 = []
for caso in casos_prueba:
    for t in DIAS_A_EVALUAR:
        res = evaluar_score_compuesto_v2(caso["id_parcela"], caso["sos_fecha"], t, df_evi, mu_evi_grano, rango_pendiente_por_dia[t])
        filas_v2.append({"id_parcela": caso["id_parcela"], "grupo": caso["grupo"], "dia": t, **res})

df_score_v2 = pd.DataFrame(filas_v2)
df_score_v2.pivot_table(index=["grupo", "id_parcela"], columns="dia", values="score_compuesto").round(1)

Rango de pendiente (min, max) de los 4 confirmados de grano, por día:
{30: (np.float64(0.0032009252331154066), np.float64(0.014449932901316223)), 42: (np.float64(0.0028560142241856375), np.float64(0.01319020492078179)), 60: (np.float64(0.0025529581482735863), np.float64(0.009893321356327638))}


dia                            30    42    60
grupo            id_parcela                  
control_negativo 0           38.5   8.3   0.0
                 12          30.3  25.6  18.3
                 13           0.0   0.0   0.0
grano_confirmado 10          25.2  42.0  46.9
                 11          34.8  51.2  58.9
                 14          99.8  99.5  99.4
                 18           5.4   5.2   6.4
lento_heldout    15           0.0   0.0  10.9
                 16          15.1  12.6   8.0

## Clasificador Dual V3

In [ ]:
def cargar_patron_desde_bd(conn, subtipo, version=None):
    query = """
        SELECT dia_post_sos, evi_promedio
        FROM patron_referencia_fenologico
        WHERE subtipo = ?
        {}
        ORDER BY dia_post_sos
    """.format("AND version = ?" if version else
               "AND version = (SELECT MAX(version) FROM patron_referencia_fenologico WHERE subtipo = ?)")
    params = (subtipo, version) if version else (subtipo, subtipo)
    df = pd.read_sql(query, conn, params=params)
    return df["evi_promedio"].values

In [ ]:
# Celda — Persistir los dos patrones de referencia a la base de datos

def persistir_patron(conn, subtipo, mu_evi, ids_usados, matriz_evi, version=1):
    evi_desviacion = np.std(matriz_evi, axis=0)  # dispersión por día, útil para inspección futura
    ids_str = ",".join(str(i) for i in sorted(ids_usados))
    n_muestras = len(ids_usados)

    registros = [
        (subtipo, dia, float(mu_evi[dia]), float(evi_desviacion[dia]), n_muestras, ids_str, version)
        for dia in range(len(mu_evi))
    ]

    with conn:
        conn.execute(
            "DELETE FROM patron_referencia_fenologico WHERE subtipo = ? AND version = ?",
            (subtipo, version),
        )
        conn.executemany(
            """
            INSERT INTO patron_referencia_fenologico
                (subtipo, dia_post_sos, evi_promedio, evi_desviacion, n_muestras, ids_parcelas_usadas, version)
            VALUES (?, ?, ?, ?, ?, ?, ?)
            """,
            registros,
        )

persistir_patron(conn, "grano_rapido", mu_evi_rapido, ids_rapido, matriz_rapido, version=1)
persistir_patron(conn, "grano_lento", mu_evi_lento, ids_lento, matriz_lento, version=1)

print("Patrones persistidos:")
print(pd.read_sql(
    "SELECT subtipo, COUNT(*) AS dias, MAX(n_muestras) AS n_muestras, MAX(ids_parcelas_usadas) AS ids FROM patron_referencia_fenologico GROUP BY subtipo",
    conn,
))

In [ ]:
# Celda — Persistir los scores calculados a clasificacion_cultivo_ciclo

def persistir_clasificacion(conn, tabla_scores, tabla_ciclos_final, dia_ventana, umbral_maiz=70, umbral_no_maiz=30):
    registros = []
    for _, row in tabla_scores[tabla_scores["dia"] == dia_ventana].iterrows():
        ciclo = tabla_ciclos_final[
            (tabla_ciclos_final["id_parcela"] == row["id_parcela"])
        ].iloc[0]  # ajustar si hay más de un ciclo por parcela en tabla_scores

        score = row["score_final"] if "score_final" in row else row["score_compuesto"]
        if pd.isna(score):
            cultivo_predicho = "incierto"
        elif score >= umbral_maiz:
            cultivo_predicho = "maiz"
        elif score <= umbral_no_maiz:
            cultivo_predicho = "no_maiz"
        else:
            cultivo_predicho = "incierto"

        ventana = "T1" if dia_ventana <= 30 else ("T2" if dia_ventana <= 60 else "T3")
        patron_usado = row.get("mejor_patron", "grano_rapido")

        registros.append((
            None,  # id_ciclo -- pendiente de resolver contra produccion_acumulada_ciclo real
            row["id_parcela"], ventana, int(dia_ventana), patron_usado,
            None, None, float(score), cultivo_predicho,
        ))

    with conn:
        conn.executemany(
            """
            INSERT INTO clasificacion_cultivo_ciclo
                (id_ciclo, id_parcela, ventana, dia_post_sos, patron_usado,
                 score_forma_pearson, score_magnitud_pendiente, score_compuesto, cultivo_predicho)
            VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)
            ON CONFLICT (id_ciclo, ventana) DO UPDATE SET
                score_compuesto = excluded.score_compuesto,
                cultivo_predicho = excluded.cultivo_predicho,
                fecha_calculo = CURRENT_TIMESTAMP
            """,
            registros,
        )

persistir_clasificacion(conn, df_score_v3.reset_index(), tabla_ciclos_final, dia_ventana=42)

In [22]:
# Celda — Dos patrones (rápido y lento) + normalización por mediana en vez de min-max

# Reclasificación: 18 pasa al grupo lento
ciclos_confirmados_grano = [
    {"id_parcela": 10, "año": 2025, "mes": 8},
    {"id_parcela": 11, "año": 2025, "mes": 8},
    {"id_parcela": 14, "año": 2025, "mes": 6},
]
ciclos_confirmados_lento = [
    {"id_parcela": 15, "año": 2025, "mes": 7},
    {"id_parcela": 16, "año": 2025, "mes": 7},
    {"id_parcela": 18, "año": 2025, "mes": 4},
]

tabla_ciclos_final["subtipo_maiz"] = "desconocido"
for ref in ciclos_confirmados_grano:
    mask = ((tabla_ciclos_final["id_parcela"] == ref["id_parcela"]) &
            (tabla_ciclos_final["sos_fecha"].dt.year == ref["año"]) &
            (tabla_ciclos_final["sos_fecha"].dt.month == ref["mes"]) &
            (tabla_ciclos_final["es_confirmado_maiz"]))
    tabla_ciclos_final.loc[mask, "subtipo_maiz"] = "grano_rapido"
for ref in ciclos_confirmados_lento:
    mask = ((tabla_ciclos_final["id_parcela"] == ref["id_parcela"]) &
            (tabla_ciclos_final["sos_fecha"].dt.year == ref["año"]) &
            (tabla_ciclos_final["sos_fecha"].dt.month == ref["mes"]) &
            (tabla_ciclos_final["es_confirmado_maiz"]))
    tabla_ciclos_final.loc[mask, "subtipo_maiz"] = "grano_lento"

def construir_patron(nombre_subtipo, tabla, df_evi, ventana=60):
    ref = tabla[tabla["subtipo_maiz"] == nombre_subtipo]
    matriz, ids = extraer_matrices_por_ciclo(df_evi, ref, ventana=ventana)
    return matriz, ids, np.mean(matriz, axis=0)

matriz_rapido, ids_rapido, mu_evi_rapido = construir_patron("grano_rapido", tabla_ciclos_final, df_evi)
matriz_lento, ids_lento, mu_evi_lento = construir_patron("grano_lento", tabla_ciclos_final, df_evi)

print(f"Patrón rápido: {ids_rapido}")
print(f"Patrón lento: {ids_lento}")

def rango_pendiente_por_dia(matriz_ids, tabla, df_evi, dias):
    out = {}
    for t in dias:
        vals = []
        for id_p in matriz_ids:
            sos = tabla[tabla["id_parcela"] == id_p]["sos_fecha"].iloc[0]
            vals.append(pendiente_verdeo_truncada(id_p, sos, t, df_evi))
        out[t] = (np.median(vals), vals)  # guardamos también los valores crudos para inspección
    return out

# Normalización por mediana con banda de tolerancia (no min-max)
def evaluar_score_v3(id_parcela, sos, t_actual, df_evi, mu_ref, mediana_pendiente_ref, tolerancia=0.5):
    r = correlacion_truncada(id_parcela, sos, t_actual, mu_ref, df_evi)
    pend_obs = pendiente_verdeo_truncada(id_parcela, sos, t_actual, df_evi)
    if np.isnan(r) or np.isnan(pend_obs) or mediana_pendiente_ref <= 0:
        return {"r_forma": np.nan, "pendiente_obs": np.nan, "score_compuesto": np.nan}
    # ratio contra la mediana, capado en 1.0; tolerancia define qué tan por debajo de la mediana aún es aceptable
    ratio = min(1.0, max(0.0, (pend_obs / mediana_pendiente_ref - tolerancia) / (1 - tolerancia)))
    score = max(0.0, r) * ratio * 100
    return {"r_forma": r, "pendiente_obs": pend_obs, "score_compuesto": score}

medianas_rapido = rango_pendiente_por_dia(ids_rapido, tabla_ciclos_final, df_evi, DIAS_A_EVALUAR)
medianas_lento = rango_pendiente_por_dia(ids_lento, tabla_ciclos_final, df_evi, DIAS_A_EVALUAR)

filas_v3 = []
for caso in casos_prueba:
    for t in DIAS_A_EVALUAR:
        s_rapido = evaluar_score_v3(caso["id_parcela"], caso["sos_fecha"], t, df_evi, mu_evi_rapido, medianas_rapido[t][0])
        s_lento = evaluar_score_v3(caso["id_parcela"], caso["sos_fecha"], t, df_evi, mu_evi_lento, medianas_lento[t][0])
        mejor = "rapido" if (s_rapido["score_compuesto"] or 0) >= (s_lento["score_compuesto"] or 0) else "lento"
        score_final = max(s_rapido["score_compuesto"] or 0, s_lento["score_compuesto"] or 0)
        filas_v3.append({"id_parcela": caso["id_parcela"], "grupo": caso["grupo"], "dia": t,
                          "score_rapido": s_rapido["score_compuesto"], "score_lento": s_lento["score_compuesto"],
                          "mejor_patron": mejor, "score_final": score_final})

df_score_v3 = pd.DataFrame(filas_v3)
df_score_v3.pivot_table(index=["grupo", "id_parcela"], columns="dia", values="score_final").round(1)

Patrón rápido: [10, 11, 14]
Patrón lento: [15, 16, 18]


dia                            30    42    60
grupo            id_parcela                  
control_negativo 0           98.0  66.7   0.0
                 12          98.7  97.5  96.8
                 13           0.0   0.0   0.0
grano_confirmado 10          99.6  99.9  99.5
                 11          99.7  99.9  99.7
                 14          99.7  99.4  99.4
                 18          99.2  97.6  98.6
lento_heldout    15           0.0  34.9  97.7
                 16          98.2  95.9  97.9

## Celdas de trabajo futuro

In [ ]:
# ==============================================================================
# CELDA 1: CONSTRUCCIÓN DEL PATRÓN MAESTRO DE MAÍZ EN GRANO
# ==============================================================================

import numpy as np
import pandas as pd

# 1. Definir tus ciclos reales confirmados de grano
ciclos_confirmados_grano = [
    {"id_parcela": 10, "año": 2025, "mes": 8},
    {"id_parcela": 11, "año": 2025, "mes": 8},
    {"id_parcela": 14, "año": 2025, "mes": 6},
    {"id_parcela": 18, "año": 2025, "mes": 4}
]

# 2. Inicializar columna e inyectar la etiqueta en la tabla consolidada
tabla_ciclos_final["subtipo_maiz"] = "desconocido"

for ref in ciclos_confirmados_grano:
    mask = (
        (tabla_ciclos_final["id_parcela"] == ref["id_parcela"]) & 
        (tabla_ciclos_final["sos_fecha"].dt.year == ref["año"]) & 
        (tabla_ciclos_final["sos_fecha"].dt.month == ref["mes"]) &
        (tabla_ciclos_final["es_confirmado_maiz"] == True)
    )
    tabla_ciclos_final.loc[mask, "subtipo_maiz"] = "grano"

# 3. Filtrar la tabla para quedarnos ÚNICAMENTE con tus referencias de grano
ciclos_referencia_grano = tabla_ciclos_final[tabla_ciclos_final["subtipo_maiz"] == "grano"]

# 4. Función robusta para extraer los vectores alineados (t=0 es el SOS de cada ciclo)
def extraer_matrices_por_ciclo(df_indice, tabla_filtrada, ventana=60):
    matrices = []
    for idx, ciclo in tabla_filtrada.iterrows():
        id_p = ciclo["id_parcela"]
        col = f"id_{id_p}"
        sos = ciclo["sos_fecha"]
        
        # Generar rango de fechas consecutivo desde el SOS específico de este ciclo
        rango_fechas = pd.date_range(start=sos, periods=ventana + 1, freq='D')
        
        try:
            # Extraer los datos usando el índice temporal
            serie_alineada = df_indice.loc[rango_fechas, col].values
                
            matrices.append(serie_alineada)
        except KeyError:
            print(f"⚠️ Alerta: No se encontraron datos para la columna '{col}' en el rango del SOS {sos.strftime('%Y-%m-%d')}")
            continue
            
    return np.array(matrices)

# 5. Extraer matrices base (Dimensiones esperadas: 4 parcelas x 61 días)
matriz_evi_grano = extraer_matrices_por_ciclo(df_evi, ciclos_referencia_grano, ventana=60)
matriz_lswi_grano = extraer_matrices_por_ciclo(df_lswi, ciclos_referencia_grano, ventana=60)

# 6. Calcular perfiles estadísticos maestros (Media y Desviación Estándar)
mu_evi_grano = np.mean(matriz_evi_grano, axis=0)
sigma_evi_grano = np.std(matriz_evi_grano, axis=0)

mu_lswi_grano = np.mean(matriz_lswi_grano, axis=0)
sigma_lswi_grano = np.std(matriz_lswi_grano, axis=0)

print(f"--- PROCESO COMPLETADO ---")
print(f"Ciclos identificados en tabla final: {len(ciclos_referencia_grano)}")
print(f"Dimensión de la matriz EVI de referencia: {matriz_evi_grano.shape}")
print(f"Dimensión de la matriz LSWI de referencia: {matriz_lswi_grano.shape}")

--- PROCESO COMPLETADO ---
Ciclos identificados en tabla final: 4
Dimensión de la matriz EVI de referencia: (4, 61)
Dimensión de la matriz LSWI de referencia: (4, 61)


In [19]:
# ==============================================================================
# CELDA 3: ALGORITMO CORE TF-PHS
# ==============================================================================
from scipy.stats import pearsonr

def evaluar_parcela_tf_phs(id_parcela, fecha_sos, t_actual, df_evi, df_lswi):
    """
    Ejecuta el score de similitud feno-hidrológica para una ventana truncada (t_actual <= 60).
    """
    # Máximos históricos regionales (Paso 1: Escalado de Amplitud Absoluta)
    EVI_MAX_REGIONAL = 0.82
    LSWI_MAX_REGIONAL = 0.29
    
    # 1. Extraer ventana temporal observada de la parcela incógnita
    col = f"id_{id_parcela}"
    rango_fechas = pd.date_range(start=fecha_sos, periods=t_actual + 1, freq='D')
    
    try:
        evi_obs = df_evi.loc[rango_fechas, col].values / EVI_MAX_REGIONAL
        lswi_obs = df_lswi.loc[rango_fechas, col].values / LSWI_MAX_REGIONAL
    except KeyError:
        return {"tipo": "Desconocido", "score": 0.0, "alerta": "Sin Datos", "accion": "Revisar ID"}
    
    # 2. Truncar patrones maestros al día actual evaluado
    ref_evi_grano_trunc = mu_evi_grano[:t_actual + 1] / EVI_MAX_REGIONAL
    # ref_evi_elote_trunc = mu_evi_elote[:t_actual + 1] / EVI_MAX_REGIONAL
    
    # Paso 2: Cálculo del Ranking de Forma (EVI) usando Pearson
    # Manejo de seguridad por si no hay varianza (ej. datos planos)
    if np.std(evi_obs) == 0:
        r_grano, r_elote = 0, 0
    else:
        r_grano, _ = pearsonr(evi_obs, ref_evi_grano_trunc)
        # r_elote, _ = pearsonr(evi_obs, ref_evi_elote_trunc)
    
    # Evitar nan en las correlaciones si las ventanas son extremadamente cortas
    r_grano = 0 if np.isnan(r_grano) else r_grano
    r_elote = 0 if np.isnan(r_elote) else r_elote
    
    # Seleccionar el mejor ajuste de forma geométrica
    if r_grano >= r_elote:
        tipo_detectado = "Grano (Híbrido)"
        r_max = r_grano
        # Asignar bandas correspondientes de LSWI
        mu_lswi_ref = mu_lswi_grano[t_actual] / LSWI_MAX_REGIONAL
        sigma_lswi_ref = sigma_lswi_grano[t_actual] / LSWI_MAX_REGIONAL
    # else:
    #     tipo_detectado = "Elotero (Precoz)"
    #     r_max = r_elote
    #     mu_lswi_ref = mu_lswi_elote[t_actual] / LSWI_MAX_REGIONAL
    #     sigma_lswi_ref = sigma_lswi_elote[t_actual] / LSWI_MAX_REGIONAL

    # Paso 3: Score de Probabilidad Continua Inicial
    score_similitud = max(0.0, r_max * 100)
    
    # Paso 4: Monitoreo Hidrológico y Penalización (LSWI)
    banda_inferior_lswi = mu_lswi_ref - (2 * sigma_lswi_ref)
    lswi_actual = lswi_obs[-1]
    
    estado_alerta = "👍 Óptimo"
    accion_sistema = "Mantener en monitoreo alto" if "Grano" in tipo_detectado else "Alta probabilidad de cosecha temprana"
    
    if lswi_actual < banda_inferior_lswi:
        score_similitud = score_similitud * 0.80 # Penalización del 20%
        estado_alerta = "⚠️ Alerta: Anomalía LSWI"
        accion_sistema = f"Caída de humedad detectada en día {t_actual} post-SOS"
        
    # Clasificación dura final para el operador en base a umbral crítico
    if score_similitud < 70.0:
        tipo_detectado = "Desconocido"
        estado_alerta = "❌ No Maíz"
        accion_sistema = "Descartar o verificación de campo"

    return {
        "Tipo Detectado": tipo_detectado,
        "Score Probabilidad": f"{score_similitud:.1f}%",
        "Estado Alerta": estado_alerta,
        "Acción del Sistema": accion_sistema
    }

In [23]:
# ==============================================================================
# CELDA 3: ALGORITMO CORE TF-PHS (FOCALIZADO EN PATRÓN DE GRANO)
# ==============================================================================
from scipy.stats import pearsonr

def evaluar_parcela_tf_phs_grano(id_parcela, fecha_sos, t_actual, df_evi, df_lswi):
    """
    Ejecuta el score de similitud feno-hidrológica para una ventana truncada (t_actual <= 60)
    utilizando exclusivamente el patrón maestro de Maíz en Grano.
    """
    # 1. Parámetros de Control y Escalado de Amplitud Absoluta
    EVI_MAX_REGIONAL = 0.82
    LSWI_MAX_REGIONAL = 0.29  # Ajustado al valor real de tu set de datos
    
    col = f"id_{id_parcela}"
    rango_fechas = pd.date_range(start=fecha_sos, periods=t_actual + 1, freq='D')
    
    # 2. Extracción y control de existencia de la serie observada
    try:
        evi_obs = df_evi.loc[rango_fechas, col].values / EVI_MAX_REGIONAL
        lswi_obs = df_lswi.loc[rango_fechas, col].values / LSWI_MAX_REGIONAL
    except KeyError:
        return {
            "Tipo Detectado": "Desconocido",
            "Score Probabilidad": "0.0%",
            "Estado Alerta": "⚠️ Sin Datos",
            "Acción del Sistema": f"ID {col} o rango de fechas no disponible en DataFrames"
        }
    
    # 3. Truncar el patrón maestro de grano al día actual evaluado
    ref_evi_grano_trunc = mu_evi_grano[:t_actual + 1] / EVI_MAX_REGIONAL
    
    # Paso 2: Cálculo del Ranking de Forma (EVI) usando Pearson
    if np.std(evi_obs) == 0:
        r_grano = 0.0
    else:
        r_grano, _ = pearsonr(evi_obs, ref_evi_grano_trunc)
    
    # Control de seguridad para correlaciones NaN por ventanas críticas o planas
    r_max = 0.0 if np.isnan(r_grano) else r_grano
    
    # Al evaluar solo grano, el tipo asignado por defecto inicial es Grano
    tipo_detectado = "Grano (Híbrido)"
    
    # Extraer parámetros de la banda de tolerancia hídrica (LSWI) correspondientes al día t
    mu_lswi_ref = mu_lswi_grano[t_actual] / LSWI_MAX_REGIONAL
    sigma_lswi_ref = sigma_lswi_grano[t_actual] / LSWI_MAX_REGIONAL

    # Paso 3: Score de Probabilidad Continua Inicial
    score_similitud = max(0.0, r_max * 100)
    
    # Paso 4: Monitoreo Hidrológico y Penalización (LSWI)
    banda_inferior_lswi = mu_lswi_ref - (2 * sigma_lswi_ref)
    lswi_actual = lswi_obs[-1]
    
    estado_alerta = "👍 Óptimo"
    accion_sistema = "Mantener en monitoreo alto"
    
    # Verificar si existe estrés hídrico por debajo de las 2 desviaciones estándar
    if lswi_actual < banda_inferior_lswi:
        score_similitud = score_similitud * 0.80  # Penalización estricta del 20%
        estado_alerta = "⚠️ Alerta: Anomalía LSWI"
        accion_sistema = f"Caída de humedad crítica detectada en día {t_actual} post-SOS"
        
    # Clasificación dura final (Filtro Drástico de No Maíz)
    if score_similitud < 70.0:
        tipo_detectado = "Desconocido"
        estado_alerta = "❌ No Maíz"
        accion_sistema = "Descartar o verificación física en Comayagua"

    return {
        "Tipo Detectado": tipo_detectado,
        "Score Probabilidad": f"{score_similitud:.1f}%",
        "Estado Alerta": estado_alerta,
        "Acción del Sistema": accion_sistema
    }

In [24]:
# ==============================================================================
# CELDA 4: PIPELINE DE SIMULACIÓN OPERATIVA (AUTOMÁTICO SOBRE INCÓGNITAS)
# ==============================================================================

# 1. Filtrar de la tabla final ÚNICAMENTE las parcelas que son incógnitas ("desconocido")
# Esto excluye automáticamente tus patrones maestros de grano y elote
df_incognitas = tabla_ciclos_final[tabla_ciclos_final["subtipo_maiz"] == "desconocido"]

# 2. Definir el día de la ventana fenológica que deseas evaluar (ej. Día 42 post-SOS)
# Puedes cambiar este valor entre 30 y 60 según el momento de la temporada que evalúes
DIA_ACTUAL_VENTANA = 42 

resultados_operador = []

print(f"Buscando parcelas incógnitas... Se encontraron {len(df_incognitas)} ciclos para evaluar.")
print(f"Ejecutando screening TF-PHS enfocado en el Día {DIA_ACTUAL_VENTANA} post-SOS...\n")

# 3. Iterar dinámicamente sobre los registros no confirmados
for idx, fila in df_incognitas.iterrows():
    id_p = fila["id_parcela"]
    sos = fila["sos_fecha"]
    
    # Ejecutar el Core matemático (construido en la Celda 3)
    res = evaluar_parcela_tf_phs_grano(
        id_parcela=id_p,
        fecha_sos=sos,
        t_actual=DIA_ACTUAL_VENTANA,
        df_evi=df_evi,
        df_lswi=df_lswi
    )
    
    # Consolidar metadatos de la tabla original con los resultados del clasificador
    registro_salida = {
        "ID Parcela": f"P-{id_p}",
        "Fecha SOS Real": sos.strftime('%Y-%m-%d'),
        "Duración Estimada (Días)": fila["duracion_dias"],
        "Tipo Detectado": res["Tipo Detectado"],
        "Score Probabilidad": res["Score Probabilidad"],
        "Estado Alerta": res["Estado Alerta"],
        "Acción del Sistema": res["Acción del Sistema"]
    }
    resultados_operador.append(registro_salida)

# 4. Construir el DataFrame del Dashboard Operativo
df_dashboard = pd.DataFrame(resultados_operador)

# 5. Ordenar por Score de Probabilidad (Descendente) para priorizar la atención del usuario
# Convertimos temporalmente a flotante para ordenar correctamente
df_dashboard["_sort_score"] = df_dashboard["Score Probabilidad"].str.rstrip('%').astype('float')
df_dashboard = df_dashboard.sort_values(by="_sort_score", ascending=False).drop(columns=["_sort_score"])

# Mostrar los resultados en la interfaz del cuaderno
print(f"📊 PANEL DE CONTROL DEL OPERADOR - ANÁLISIS DE INCÓGNITAS (T = {DIA_ACTUAL_VENTANA})")
display(df_dashboard)

Buscando parcelas incógnitas... Se encontraron 41 ciclos para evaluar.
Ejecutando screening TF-PHS enfocado en el Día 42 post-SOS...

📊 PANEL DE CONTROL DEL OPERADOR - ANÁLISIS DE INCÓGNITAS (T = 42)


,ID Parcela,Fecha SOS Real,Duración Estimada (Días),Tipo Detectado,Score Probabilidad,Estado Alerta,Acción del Sistema
14,P-7,2025-03-01,105,Grano (Híbrido),99.9%,👍 Óptimo,Mantener en monitoreo alto
27,P-10,2026-02-01,109,Grano (Híbrido),99.8%,👍 Óptimo,Mantener en monitoreo alto
19,P-8,2025-07-03,95,Grano (Híbrido),99.6%,👍 Óptimo,Mantener en monitoreo alto
18,P-8,2025-02-21,112,Grano (Híbrido),99.6%,👍 Óptimo,Mantener en monitoreo alto
28,P-11,2025-03-27,147,Grano (Híbrido),99.6%,👍 Óptimo,Mantener en monitoreo alto
25,P-9,2026-02-04,95,Grano (Híbrido),99.4%,👍 Óptimo,Mantener en monitoreo alto
11,P-5,2025-12-13,119,Grano (Híbrido),99.4%,👍 Óptimo,Mantener en monitoreo alto
22,P-9,2025-02-24,102,Grano (Híbrido),99.3%,👍 Óptimo,Mantener en monitoreo alto
29,P-11,2026-01-20,112,Grano (Híbrido),99.2%,👍 Óptimo,Mantener en monitoreo alto
23,P-9,2025-06-25,97,Grano (Híbrido),99.1%,👍 Óptimo,Mantener en monitoreo alto
